In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!nvidia-smi topo -m

	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks


In [3]:
%%writefile ddp_time.py
import os, sys, time, torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.algorithms.ddp_comm_hooks import default_hooks

batch = int(sys.argv[1]); compress = len(sys.argv) > 2
dist.init_process_group("nccl")
rank = dist.get_rank(); world = dist.get_world_size()
torch.cuda.set_device(rank)

model = nn.Sequential(nn.Linear(1024, 4096), nn.ReLU(),
                      nn.Linear(4096, 4096), nn.ReLU(),
                      nn.Linear(4096, 10)).cuda()
model = DDP(model, device_ids=[rank])
if compress:
    model.register_comm_hook(None, default_hooks.fp16_compress_hook)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(batch, 1024, device="cuda")
y = torch.randint(0, 10, (batch,), device="cuda")

for i in range(60):
    if i == 10:
        torch.cuda.synchronize(); t = time.time()
    opt.zero_grad()
    nn.functional.cross_entropy(model(x), y).backward()
    opt.step()
torch.cuda.synchronize()
if rank == 0:
    print(f"GPUs={world} batch={batch} compress={compress}: "
          f"{50 * batch * world / (time.time() - t):,.0f} samples/s")
dist.destroy_process_group()

Writing ddp_time.py


In [4]:
!torchrun --nproc_per_node=1 ddp_time.py 256
!torchrun --nproc_per_node=2 ddp_time.py 256
!torchrun --nproc_per_node=1 ddp_time.py 2048
!torchrun --nproc_per_node=2 ddp_time.py 2048
!torchrun --nproc_per_node=2 ddp_time.py 2048 compress

GPUs=1 batch=256 compress=False: 25,319 samples/s
W1115 00:01:45.288000 124 torch/distributed/run.py:851] 
W1115 00:01:45.288000 124 torch/distributed/run.py:851] *****************************************
W1115 00:01:45.288000 124 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1115 00:01:45.288000 124 torch/distributed/run.py:851] *****************************************
GPUs=2 batch=256 compress=False: 18,910 samples/s
GPUs=1 batch=2048 compress=False: 35,400 samples/s
W1115 00:02:04.766000 171 torch/distributed/run.py:851] 
W1115 00:02:04.766000 171 torch/distributed/run.py:851] *****************************************
W1115 00:02:04.766000 171 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded

In [5]:
%%writefile fsdp_mem.py
import sys, torch, torch.nn as nn, torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import ModuleWrapPolicy

mode = sys.argv[1]
dist.init_process_group("nccl")
rank = dist.get_rank(); torch.cuda.set_device(rank)

layers = [nn.Linear(4096, 4096) for _ in range(8)]
model = nn.Sequential(*layers).cuda()
if mode == "whole":
    model = FSDP(model)
elif mode == "layer":
    model = FSDP(model, auto_wrap_policy=ModuleWrapPolicy({nn.Linear}))

opt = torch.optim.Adam(model.parameters())
x = torch.randn(64, 4096, device="cuda")
torch.cuda.reset_peak_memory_stats()
for _ in range(3):
    opt.zero_grad(); model(x).sum().backward(); opt.step()
if rank == 0:
    print(f"{mode}: peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB per GPU")
dist.destroy_process_group()

Writing fsdp_mem.py


In [6]:
!torchrun --nproc_per_node=2 fsdp_mem.py none
!torchrun --nproc_per_node=2 fsdp_mem.py whole
!torchrun --nproc_per_node=2 fsdp_mem.py layer

W1115 00:02:28.168000 225 torch/distributed/run.py:851] 
W1115 00:02:28.168000 225 torch/distributed/run.py:851] *****************************************
W1115 00:02:28.168000 225 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1115 00:02:28.168000 225 torch/distributed/run.py:851] *****************************************
none: peak 2.70 GB per GPU
W1115 00:02:38.153000 245 torch/distributed/run.py:851] 
W1115 00:02:38.153000 245 torch/distributed/run.py:851] *****************************************
W1115 00:02:38.153000 245 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1115 00:02:38.153000 245 torch/d